# 01 — Instrumentation gate

**Stage:** Proposal Stage 1, second half — *"make sure we can hook into the AHN
output/state correctly"* (Gautam). This is the gate the 18 Aug pilot did not pass.

### What "correctly" means, precisely

The pilot verified that the NOWRITE hook makes the captured AHN vector exactly zero.
That is a check on hook *ordering*; it says nothing about whether the captured vector is
the thing we think it is. The claim that actually needs testing is an identity that
falls straight out of `qwen2_ahn.py`:

$$\text{resid}_{\text{AHN}}(L) - \text{resid}_{\text{NOWRITE}}(L) \;=\; W_{o}^{(L)}\,\text{ahn\_raw}(L)$$

for the **first** AHN layer. If that holds, the hook plus `o_proj` really is the memory's
contribution to the residual stream, and the J-lens has a well-typed input. At deeper
layers the two runs have already diverged upstream, so the identity is expected to
fail there — that is not a bug, it is why the check is defined on the first layer.

### Two consequences that change the experimental design

1. **Use `o_t`, never `ahn_raw`.** `AHNProbe.Capture.o_t()` applies `o_proj`.
2. **Control C1 does not live on `o_t`.** Under NOWRITE the AHN output is identically
   zero, so `Δ = o_t(AHN) − o_t(NOWRITE)` collapses to `o_t` and the control is
   vacuous — the pilot's own markdown notices this and proceeds anyway. C1 has to be
   run on the **residual stream** at the same position.


In [1]:
# --- bootstrap -------------------------------------------------------------------
# Upload `ahn_interp.py` next to this notebook (or anywhere up the tree).
import os, sys, json, importlib

def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Upload it into this notebook's directory "
               "(Jupyter: Upload button, top right of the file browser).")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Pin the working directory to wherever ahn_interp.py actually lives (normally the
# repo root). Without this, relative paths in CFG (results_dir, configs/*.json) resolve
# against whatever directory Jupyter happened to open in -- e.g. running this notebook
# from inside notebooks/ silently writes results to notebooks/results/... instead of
# results/... at the repo root, which is where every other notebook and 05's analysis
# step expect to find them.
os.chdir(_root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)
print("working directory pinned to", os.getcwd())


ahn_interp loaded from /home/jupyter-dphs-4ca1/AHN
working directory pinned to /home/jupyter-dphs-4ca1/AHN


In [2]:
# --- experiment configuration ----------------------------------------------------
# Everything that changes what a number MEANS lives here and gets saved with the run.
CFG = dict(
    model_path      = "/home/jupyter-dphs-4ca1/AHN/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
    cell            = "GatedDeltaNet",     # GatedDeltaNet | DeltaNet | Mamba2
    scale           = "3B",
    sliding_window  = 8064,                # proposal value; upstream eval uses 8064
    num_attn_sinks  = 128,                 # upstream eval default. NOT zero.
    attn_impl       = "flash_attention_2", # "eager" on T4/P100 (no Ampere -> no FA2)
    dtype           = "bfloat16",          # "float16" on T4/P100
    results_dir     = "results/run_3b_gdn",
)
ai.set_results_dir(CFG["results_dir"])
print(json.dumps(CFG, indent=2))


{
  "model_path": "/home/jupyter-dphs-4ca1/AHN/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
  "cell": "GatedDeltaNet",
  "scale": "3B",
  "sliding_window": 8064,
  "num_attn_sinks": 128,
  "attn_impl": "flash_attention_2",
  "dtype": "bfloat16",
  "results_dir": "results/run_3b_gdn"
}


In [3]:
import torch
bundle = ai.load_ahn_model(
    CFG["model_path"], dtype=getattr(torch, CFG["dtype"]),
    attn_implementation=CFG["attn_impl"],
    sliding_window=CFG["sliding_window"], num_attn_sinks=CFG["num_attn_sinks"],
)
tok, probe = bundle.tokenizer, ai.AHNProbe(bundle)
first_ahn = bundle.ahn_layers[0]
print("first AHN layer:", first_ahn, "| all:", bundle.ahn_layers[:6], "...")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

first AHN layer: 0 | all: [0, 1, 2, 3, 4, 5] ...


## Gate A — the isolation identity

In [4]:
spec = ai.build_niah_prompt(tok, "Paris", bundle, eviction_distance=1024)
inputs = tok(spec["prompt"], return_tensors="pt").to(bundle.model.device)
assert spec["ahn_will_activate"], "prompt too short for AHN to activate"

gateA = probe.verify_isolation(inputs, layer=first_ahn)

# for contrast, a deeper layer — this is EXPECTED to fail
gateA_deep = probe.verify_isolation(inputs, layer=bundle.ahn_layers[len(bundle.ahn_layers)//2])


--- isolation check, layer 0 (FIRST AHN layer) ---
  layer                    0
  is_first_ahn_layer       True
  n_tokens                 9252
  ahn_active               True
  nowrite_o_t_is_zero      True
  ahn_raw_norm             0.5769657492637634
  o_t_norm                 0.6288129687309265
  residual_norm            10.402517318725586
  relative_error           0.042794932994026105
  cosine                   0.999083936214447
  passed                   True
  => PASS
--- isolation check, layer 18 (deeper layer, divergence expected) ---
  layer                    18
  is_first_ahn_layer       False
  n_tokens                 9252
  ahn_active               True
  nowrite_o_t_is_zero      True
  ahn_raw_norm             0.6935104131698608
  o_t_norm                 0.8532447814941406
  residual_norm            62.44887161254883
  relative_error           0.18740204380390124
  cosine                   0.9829652309417725
  passed                   False
  => FAIL


In [5]:
assert gateA["passed"], (
    "GATE A FAILED. The hook output does not reconstruct the residual-stream delta. "
    "Check: (a) use_ahn_router in the audit — a learned gate breaks the plain-sum "
    "identity; (b) o_proj bias; (c) that NOWRITE really suppresses writes rather than "
    "only zeroing the captured tensor."
)
print("GATE A PASSED — o_proj(hook output) is the memory's residual-stream contribution.")


GATE A PASSED — o_proj(hook output) is the memory's residual-stream contribution.


## Gate B — NOWRITE changes the model's behaviour, not just a tensor

Zeroing a captured tensor is free. What has to be true is that suppressing writes
changes the *output distribution*. If the KL between the two next-token distributions is
~0, the memory is not doing anything at this length and every downstream number will be
noise — which is the most likely explanation for the pilot's chance-level ranks.


In [6]:
on  = probe.run(inputs, nowrite=False, capture_residual=False, keep_logits=True)
off = probe.run(inputs, nowrite=True,  capture_residual=False, keep_logits=True)

p = torch.softmax(on.logits[0, -1], dim=-1)
q = torch.softmax(off.logits[0, -1], dim=-1)
kl = float((p * (p.clamp_min(1e-12).log() - q.clamp_min(1e-12).log())).sum())
js = 0.5 * float(
    (p * (p.clamp_min(1e-12).log() - ((p+q)/2).clamp_min(1e-12).log())).sum()
    + (q * (q.clamp_min(1e-12).log() - ((p+q)/2).clamp_min(1e-12).log())).sum()
)
gateB = {
    "kl_ahn_vs_nowrite": kl,
    "js_ahn_vs_nowrite": js,
    "top1_ahn": tok.decode([int(p.argmax())]),
    "top1_nowrite": tok.decode([int(q.argmax())]),
    "top1_changed": bool(p.argmax() != q.argmax()),
    "n_tokens": on.n_tokens,
    "passed": bool(kl > 1e-3),
}
print(json.dumps(gateB, indent=2))
assert gateB["passed"], (
    "GATE B FAILED: suppressing AHN writes barely moves the output distribution. "
    "Either AHN is not active (check n_tokens vs window+sinks) or the NOWRITE hook is "
    "not on the write path. Do not proceed to J-lens work until this passes."
)
print("\nGATE B PASSED.")


{
  "kl_ahn_vs_nowrite": 0.16034048795700073,
  "js_ahn_vs_nowrite": 0.036773987114429474,
  "top1_ahn": " The",
  "top1_nowrite": " The",
  "top1_changed": false,
  "n_tokens": 9252,
  "passed": true
}

GATE B PASSED.


## Gate C — the needle is genuinely outside the lossless memory

Sanity that the *task* is what we think it is: with the needle evicted, does the model
still answer? And does answering depend on the memory? Two numbers, both cheap.


In [7]:
@torch.no_grad()
def generate(prompt, nowrite=False, max_new_tokens=12):
    ins = tok(prompt, return_tensors="pt").to(bundle.model.device)
    handles = []
    if nowrite:
        def z(m, i, o):
            return (torch.zeros_like(o[0]),) + o[1:] if isinstance(o, tuple) else torch.zeros_like(o)
        for L in bundle.ahn_layers:
            handles.append(bundle.model.model.layers[L].ahn.register_forward_hook(z))
    try:
        out = bundle.model.generate(**ins, max_new_tokens=max_new_tokens, do_sample=False,
                                    pad_token_id=tok.eos_token_id)
    finally:
        for h in handles: h.remove()
    return tok.decode(out[0, ins["input_ids"].shape[1]:], skip_special_tokens=True).strip()

ans_on  = generate(spec["prompt"], nowrite=False)
ans_off = generate(spec["prompt"], nowrite=True)
gateC = {
    "needle": spec["needle"],
    "eviction_distance": spec["actual_eviction_distance"],
    "answer_ahn": ans_on,
    "answer_nowrite": ans_off,
    "ahn_correct": spec["needle"].lower() in ans_on.lower(),
    "nowrite_correct": spec["needle"].lower() in ans_off.lower(),
}
print(json.dumps(gateC, indent=2))
print("\nInterpretation:")
print("  correct WITH memory, wrong WITHOUT  -> the ideal case: the task depends on AHN")
print("  correct in both -> the needle is reachable without memory; move it further out")
print("  wrong in both   -> the model cannot do this task at this length; easier setting")


/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.ven

{
  "needle": "Paris",
  "eviction_distance": 1043,
  "answer_ahn": "The special word in this text is \"Here we go.\"",
  "answer_nowrite": "It seems like you're referring to a pattern or rhyme that",
  "ahn_correct": false,
  "nowrite_correct": false
}

Interpretation:
  correct WITH memory, wrong WITHOUT  -> the ideal case: the task depends on AHN
  correct in both -> the needle is reachable without memory; move it further out
  wrong in both   -> the model cannot do this task at this length; easier setting


In [8]:
gates = {"gate_a_isolation": gateA, "gate_a_deep_layer_contrast": gateA_deep,
         "gate_b_behavioural": gateB, "gate_c_task": gateC,
         "prompt_spec": {k: v for k, v in spec.items() if k != "prompt"}, "cfg": CFG}
ai.save_json(gates, "01_instrumentation_gates.json")
print("saved -> 01_instrumentation_gates.json")
print("\nALL GATES:", "PASS" if (gateA["passed"] and gateB["passed"]) else "FAIL")


saved -> 01_instrumentation_gates.json

ALL GATES: PASS


### Gate for this notebook

- [ ] Gate A passes on the first AHN layer
- [ ] Gate B: KL(AHN ‖ NOWRITE) is clearly non-zero
- [ ] Gate C: you can state, in one sentence, whether the task depends on the memory

Only then does **02_jlens_fit_and_validate.ipynb** mean anything.
